# Gradient Boosting Model Selection

Fit and evaluate a leakage-safe gradient boosting classifier from `diabetes.csv`. The cleaning step from `cleaning_and_scaling.ipynb` is kept inside the pipeline, and hyperparameters are selected with stratified cross-validation.

In [1]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42

## Cleaning logic checked

This notebook uses the same cleaning idea as `cleaning_and_scaling.ipynb`: zeros are treated as invalid for `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI`, then replaced with each column median.

The median replacement happens inside the scikit-learn pipeline. During cross-validation, each fold learns those medians only from its own training data, avoiding leakage into validation or test rows.

In [2]:
raw_data = pd.read_csv("diabetes.csv")
print(raw_data.shape)
raw_data.head()

(768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
zero_as_missing_columns = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
]

raw_data.eq(0).sum()

Pregnancies                 111
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                     500
dtype: int64

In [4]:
class ZeroToMedianImputer(BaseEstimator, TransformerMixin):
    """Replace invalid zeros in selected columns with training medians."""

    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.feature_names_in_ = X.columns.to_numpy()
        self.medians_ = {}

        for column in self.columns:
            non_zero_values = X.loc[X[column] != 0, column]
            self.medians_[column] = non_zero_values.median()

        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.feature_names_in_).copy()

        for column, median in self.medians_.items():
            X[column] = X[column].replace(0, median)

        return X

In [5]:
X = raw_data.drop(columns="Outcome")
y = raw_data["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Train class balance:")
print(y_train.value_counts(normalize=True).rename("proportion"))
print("\nTest class balance:")
print(y_test.value_counts(normalize=True).rename("proportion"))

Train class balance:
0    0.651466
1    0.348534
Name: proportion, dtype: float64

Test class balance:
0    0.649351
1    0.350649
Name: proportion, dtype: float64


## Cross-validated hyperparameter search

`GridSearchCV` evaluates gradient boosting settings with 5-fold stratified CV. Scaling is not used because tree-based boosting does not require standardized features.

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

pipeline = Pipeline(
    steps=[
        ("zero_imputer", ZeroToMedianImputer(columns=zero_as_missing_columns)),
        ("model", GradientBoostingClassifier(random_state=RANDOM_STATE)),
    ]
)

param_grid = {
    "model__n_estimators": [50, 100, 200],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_depth": [1, 2, 3],
    "model__min_samples_leaf": [1, 3, 5, 10],
    "model__subsample": [0.8, 1.0],
    "model__max_features": [None, "sqrt"],
}

search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring={"roc_auc": "roc_auc", "f1": "f1", "accuracy": "accuracy"},
    refit="roc_auc",
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
)

search.fit(X_train, y_train)

print("Best CV ROC AUC:", round(search.best_score_, 4))
print("Best parameters:")
search.best_params_

Best CV ROC AUC: 0.8407
Best parameters:


{'model__learning_rate': 0.05,
 'model__max_depth': 1,
 'model__max_features': 'sqrt',
 'model__min_samples_leaf': 3,
 'model__n_estimators': 200,
 'model__subsample': 0.8}

In [7]:
cv_results = pd.DataFrame(search.cv_results_)

ranked_results = (
    cv_results[
        [
            "rank_test_roc_auc",
            "mean_test_roc_auc",
            "std_test_roc_auc",
            "mean_test_f1",
            "mean_test_accuracy",
            "param_model__n_estimators",
            "param_model__learning_rate",
            "param_model__max_depth",
            "param_model__min_samples_leaf",
            "param_model__subsample",
            "param_model__max_features",
        ]
    ]
    .sort_values("rank_test_roc_auc")
    .head(10)
)

ranked_results

,rank_test_roc_auc,mean_test_roc_auc,std_test_roc_auc,mean_test_f1,mean_test_accuracy,param_model__n_estimators,param_model__learning_rate,param_model__max_depth,param_model__min_samples_leaf,param_model__subsample,param_model__max_features
178,1,0.840717,0.018881,0.634484,0.776849,200,0.05,1,3,0.8,sqrt
290,2,0.839877,0.018229,0.641978,0.776849,100,0.10,1,1,0.8,None
172,3,0.839603,0.019029,0.638289,0.778475,200,0.05,1,1,0.8,sqrt
302,4,0.839353,0.018518,0.645286,0.778475,100,0.10,1,5,0.8,None
184,5,0.839320,0.019362,0.627133,0.773597,200,0.05,1,5,0.8,sqrt
296,6,0.838951,0.016895,0.648989,0.780101,100,0.10,1,3,0.8,None
342,7,0.838945,0.017143,0.639617,0.768693,50,0.10,2,3,0.8,None
192,8,0.838840,0.021923,0.630615,0.778475,50,0.05,2,1,0.8,None
314,9,0.838615,0.021823,0.626661,0.773597,100,0.10,1,1,0.8,sqrt
200,10,0.838470,0.019358,0.650833,0.778489,100,0.05,2,3,0.8,None


## Held-out test performance

The default 0.50 threshold is reported, and a second threshold is selected on the training set to maximize F1. The test set remains held out until this final evaluation.

In [8]:
best_model = search.best_estimator_

y_train_proba = best_model.predict_proba(X_train)[:, 1]
y_test_proba = best_model.predict_proba(X_test)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_train, y_train_proba)
f1_scores = 2 * precision * recall / (precision + recall + 1e-12)
best_threshold = thresholds[np.argmax(f1_scores[:-1])]

metrics = pd.DataFrame(
    {
        "threshold": [0.50, best_threshold],
        "roc_auc": [roc_auc_score(y_test, y_test_proba)] * 2,
        "accuracy": [
            accuracy_score(y_test, y_test_proba >= 0.50),
            accuracy_score(y_test, y_test_proba >= best_threshold),
        ],
        "f1": [
            f1_score(y_test, y_test_proba >= 0.50),
            f1_score(y_test, y_test_proba >= best_threshold),
        ],
    },
    index=["default_0.50", "train_f1_optimized"],
)

metrics

,threshold,roc_auc,accuracy,f1
default_0.50,0.500000,0.818519,0.74026,0.583333
train_f1_optimized,0.320239,0.818519,0.74026,0.677419


In [9]:
test_predictions = (y_test_proba >= best_threshold).astype(int)

print(f"Selected threshold: {best_threshold:.3f}")
print("\nConfusion matrix:")
print(confusion_matrix(y_test, test_predictions))
print("\nClassification report:")
print(classification_report(y_test, test_predictions, digits=3))

Selected threshold: 0.320

Confusion matrix:
[[72 28]
 [12 42]]

Classification report:
              precision    recall  f1-score   support

           0      0.857     0.720     0.783       100
           1      0.600     0.778     0.677        54

    accuracy                          0.740       154
   macro avg      0.729     0.749     0.730       154
weighted avg      0.767     0.740     0.746       154



## Feature importance

Gradient boosting importances show which features contributed most to the fitted boosted trees.

In [10]:
final_gb = best_model.named_steps["model"]

importance_table = pd.DataFrame(
    {
        "feature": X.columns,
        "importance": final_gb.feature_importances_,
    }
).sort_values("importance", ascending=False)

importance_table

,feature,importance
1,Glucose,0.497782
5,BMI,0.176438
7,Age,0.120728
4,Insulin,0.081560
6,DiabetesPedigreeFunction,0.056509
0,Pregnancies,0.038881
3,SkinThickness,0.019386
2,BloodPressure,0.008715


In [11]:
print("Final leakage-safe gradient boosting pipeline:")
print(best_model)

Final leakage-safe gradient boosting pipeline:
Pipeline(steps=[('zero_imputer',
                 ZeroToMedianImputer(columns=['Glucose', 'BloodPressure',
                                              'SkinThickness', 'Insulin',
                                              'BMI'])),
                ('model',
                 GradientBoostingClassifier(learning_rate=0.05, max_depth=1,
                                            max_features='sqrt',
                                            min_samples_leaf=3,
                                            n_estimators=200, random_state=42,
                                            subsample=0.8))])
